In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "dbw_company_risk"
stream_path = "abfss://raw@ukcompanyrisk2026.dfs.core.windows.net/stream/"


def j(path):
    """Pull one field out of the raw JSON text; missing fields become null."""
    return F.get_json_object(F.col("value"), path)


def blank_to_null(col):
    return F.when(col == "", None).otherwise(col)


raw = spark.read.text(stream_path).withColumn("_source_file", F.col("_metadata.file_path"))

# The API uses codes like "liquidation" or "active-proposal-to-strike-off"
status = F.lower(F.coalesce(j("$.data.company_status_detail"), j("$.data.company_status")))
status_group = (
    F.when(status == "active", "Active")
     .when(status.contains("strike-off"), "Proposal to strike off")
     .when(status.contains("liquidation"), "Liquidation")
     .when(status.contains("voluntary-arrangement"), "Voluntary arrangement")
     .when(status.contains("administration") | status.contains("receiver"), "Administration or receivership")
     .when(status.isin("dissolved", "closed", "converted-closed", "removed"), "Dissolved")
     .otherwise("Other")
)
postcode = F.upper(j("$.data.registered_office_address.postal_code"))

changes = (
    raw.where(j("$.resource_kind") == "company-profile")
    .select(
        j("$.data.company_number").alias("company_number"),
        j("$.data.company_name").alias("company_name"),
        j("$.data.type").alias("company_type"),
        status.alias("status_raw"),
        status_group.alias("status_group"),
        j("$.data.sic_codes[0]").alias("sic_code"),
        j("$.data.registered_office_address.locality").alias("post_town"),
        postcode.alias("postcode"),
        blank_to_null(F.regexp_extract(postcode, r"^([A-Z]{1,2})", 1)).alias("postcode_area"),
        F.to_date(j("$.data.date_of_creation")).alias("incorporation_date"),
        F.to_date(j("$.data.accounts.next_accounts.due_on")).alias("accounts_next_due"),
        F.to_date(j("$.data.accounts.last_accounts.made_up_to")).alias("accounts_last_made_up"),
        F.to_date(j("$.data.confirmation_statement.next_due")).alias("confirmation_next_due"),
        F.upper(F.regexp_replace(j("$.data.accounts.last_accounts.type"), "-", " ")).alias("accounts_category"),
        j("$.event.type").alias("event_type"),
        j("$.event.timepoint").cast("long").alias("timepoint"),
        F.to_timestamp(j("$.event.published_at")).alias("published_at"),
        F.col("_source_file"),
    )
    .where("company_number IS NOT NULL")
    .withColumn("status_group",
                F.when(F.col("event_type") == "deleted", "Dissolved").otherwise(F.col("status_group")))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("snapshot_date", F.to_date("published_at"))
    .withColumn("company_age_years",
                F.round(F.datediff("snapshot_date", "incorporation_date") / 365.25, 1))
    .withColumn("accounts_overdue",
                F.coalesce(F.col("accounts_next_due") < F.col("snapshot_date"), F.lit(False)))
    .withColumn("confirmation_overdue",
                F.coalesce(F.col("confirmation_next_due") < F.col("snapshot_date"), F.lit(False)))
)

# The stream only sends industry codes, so borrow the descriptions from the silver table
sic_lookup = (spark.table(f"{catalog}.silver.companies")
              .where("sic_code IS NOT NULL AND sic_description IS NOT NULL")
              .groupBy("sic_code").agg(F.first("sic_description").alias("sic_description")))
changes = changes.join(F.broadcast(sic_lookup), "sic_code", "left")
changes.createOrReplaceTempView("all_changes")

# A company can change several times; keep only its latest event
latest = (changes
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("company_number").orderBy(F.desc("timepoint"))))
    .where("rn = 1").drop("rn"))
latest.createOrReplaceTempView("latest_changes")

# Give the silver table columns that record its last change (only needed the first time)
if "last_change_timepoint" not in spark.table(f"{catalog}.silver.companies").columns:
    spark.sql(f"""ALTER TABLE {catalog}.silver.companies
                  ADD COLUMNS (last_change_timepoint BIGINT, last_change_at TIMESTAMP)""")

print(f"Events: {changes.count():,}   Companies changed: {latest.count():,}")
display(changes.groupBy("event_type", "status_group").count().orderBy(F.desc("count")))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbw_company_risk.silver.company_changes AS
SELECT * FROM all_changes WHERE false;

MERGE INTO dbw_company_risk.silver.company_changes AS h
USING all_changes AS c
ON h.company_number = c.company_number AND h.timepoint = c.timepoint
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
MERGE INTO dbw_company_risk.silver.companies AS t
USING latest_changes AS s
ON t.company_number = s.company_number
WHEN MATCHED AND s.timepoint > coalesce(t.last_change_timepoint, -1) THEN UPDATE SET
  t.company_name          = coalesce(s.company_name, t.company_name),
  t.status_raw            = s.status_raw,
  t.status_group          = s.status_group,
  t.sic_code              = coalesce(s.sic_code, t.sic_code),
  t.sic_description       = CASE WHEN s.sic_code IS NULL OR s.sic_code = t.sic_code
                                 THEN t.sic_description ELSE s.sic_description END,
  t.post_town             = coalesce(s.post_town, t.post_town),
  t.postcode              = coalesce(s.postcode, t.postcode),
  t.postcode_area         = coalesce(s.postcode_area, t.postcode_area),
  t.incorporation_date    = coalesce(s.incorporation_date, t.incorporation_date),
  t.accounts_next_due     = s.accounts_next_due,
  t.accounts_last_made_up = coalesce(s.accounts_last_made_up, t.accounts_last_made_up),
  t.confirmation_next_due = s.confirmation_next_due,
  t.accounts_category     = coalesce(s.accounts_category, t.accounts_category),
  t.snapshot_date         = s.snapshot_date,
  t.company_age_years     = coalesce(s.company_age_years, t.company_age_years),
  t.accounts_overdue      = s.accounts_overdue,
  t.confirmation_overdue  = s.confirmation_overdue,
  t._source_file          = s._source_file,
  t._ingested_at          = s._ingested_at,
  t.last_change_timepoint = s.timepoint,
  t.last_change_at        = s.published_at
WHEN NOT MATCHED THEN INSERT (
  company_number, company_name, company_type, status_raw, status_group, sic_code,
  sic_description, post_town, postcode, postcode_area, incorporation_date,
  accounts_next_due, accounts_last_made_up, confirmation_next_due, accounts_category,
  charges_outstanding, _source_file, _ingested_at, snapshot_date, company_age_years,
  accounts_overdue, confirmation_overdue, last_change_timepoint, last_change_at
) VALUES (
  s.company_number, s.company_name, s.company_type, s.status_raw, s.status_group, s.sic_code,
  s.sic_description, s.post_town, s.postcode, s.postcode_area, s.incorporation_date,
  s.accounts_next_due, s.accounts_last_made_up, s.confirmation_next_due, s.accounts_category,
  0, s._source_file, s._ingested_at, s.snapshot_date, s.company_age_years,
  s.accounts_overdue, s.confirmation_overdue, s.timepoint, s.published_at
)

In [0]:
%sql
SELECT company_number, company_name, status_group, accounts_overdue, last_change_at
FROM dbw_company_risk.silver.companies
WHERE last_change_timepoint IS NOT NULL
ORDER BY last_change_at DESC
LIMIT 20